In [1]:
!pip install -q openpyxl pandas tqdm


In [2]:
from google.colab import files
uploaded = files.upload()


Saving pershop_intent.xlsx to pershop_intent.xlsx


In [3]:
import pandas as pd

df = pd.read_excel("pershop_intent.xlsx")
df = df.dropna(subset=["text", "intent"]).reset_index(drop=True)
df["text"] = df["text"].astype(str).str.strip()
df["intent"] = df["intent"].astype(str).str.strip()

print("Number of samples:", len(df))
print("Columns:", df.columns)
print("Unique intents:", df["intent"].unique())


Number of samples: 11053
Columns: Index(['intent', 'text'], dtype='object')
Unique intents: ['Greeting' 'purchase' 'inform' 'Deny' 'User_Confirm' 'ask_detail'
 'user_ask' 'Goodby' 'chitchat' 'prev_inform' 'Regret' 'SuperProduct_ask']


In [25]:
from sklearn.model_selection import train_test_split

# 20% برای test، مثل قبل
_, test_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df["intent"]
)

print("Original test size:", len(test_df))

# حالا یک زیرمجموعه‌ی متعادل، مثلاً حدود 200 نمونه
INTENT_LABELS = sorted(df["intent"].unique().tolist())

N_SAMPLES = len(test_df)  # می‌تونی زیاد/کمش کنی

if len(test_df) <= N_SAMPLES:
    llm_df = test_df.copy()
else:
    # تا جای ممکن به‌صورت stratified از هر intent چند نمونه بگیر
    per_label = max(1, N_SAMPLES // len(INTENT_LABELS))
    llm_df = test_df.groupby("intent", group_keys=False).apply(
        lambda x: x.sample(min(len(x), per_label), random_state=42)
    ).reset_index(drop=True)

print("LLM eval size:", len(llm_df))
print("LLM intents distribution:")
print(llm_df["intent"].value_counts())


Original test size: 2211
LLM eval size: 2211
LLM intents distribution:
intent
purchase            588
User_Confirm        555
inform              506
Greeting            277
Goodby              210
Deny                 40
chitchat             14
user_ask             13
prev_inform           3
SuperProduct_ask      3
Regret                1
ask_detail            1
Name: count, dtype: int64


In [26]:
import unicodedata

def normalize_label(s: str) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    s = unicodedata.normalize("NFKC", s)
    return s

INTENT_LABELS = sorted(df["intent"].unique().tolist())
canonical_intents = {normalize_label(x): x for x in INTENT_LABELS}

print("INTENT_LABELS:", INTENT_LABELS)


INTENT_LABELS: ['Deny', 'Goodby', 'Greeting', 'Regret', 'SuperProduct_ask', 'User_Confirm', 'ask_detail', 'chitchat', 'inform', 'prev_inform', 'purchase', 'user_ask']


In [27]:
INTENT_LIST_STR = ", ".join(INTENT_LABELS)

def build_batch_prompt(utterances):
    """
    utterances: list of Persian user utterances (strings)
    returns: single prompt string
    """
    items_str = ""
    for i, utt in enumerate(utterances, start=1):
        items_str += f'{i}. "{utt}"\n'

    return f"""
You are an expert NLU system for Persian task-oriented shopping dialogues.
Your ONLY task is to classify the user's intent into EXACTLY one label.

Below are the definitions of ALL possible intents:

- Greeting: User greets the system (e.g., سلام، درود).
- Goodby: User closes the conversation (e.g., خداحافظ).
- purchase: User wants to buy a specific product or asks for a product recommendation.
- inform: User provides product attributes (brand, fat, size, price, etc.).
- user_ask: User asks a general question about products (e.g., چه برندهایی هست؟).
- ask_detail: User asks for specific product details (e.g., وزن، اندازه، قیمت دقیق).
- User_Confirm: User confirms or agrees (e.g., بله، درسته).
- Deny: User rejects or disagrees (e.g., نه، نمی‌خوام).
- chitchat: General small talk not related to shopping.
- prev_inform: User refers to previous information (e.g., همان قبلی را می‌خواهم).
- Regret: User expresses regret or apology.
- SuperProduct_ask: User asks about a domain/sub-domain (e.g., لبنیات داری؟ نوشیدنی داری؟).

Your output MUST be a JSON array. Each element must contain:
- "id": the index of the utterance (1, 2, 3, ...)
- "intent": exactly one of the labels listed above.

Rules:
- Do NOT explain your answer.
- Do NOT translate the utterances.
- Do NOT provide reasoning.
- Do NOT use any labels outside [{INTENT_LIST_STR}].

Classify the following utterances:

{items_str}

Return ONLY the JSON array.
"""


OPENAI GPT5.1 for Intent Classification

In [28]:
from google.colab import userdata  # store the key in Colab Secrets as OPENAI_API_KEY
from openai import OpenAI

# اینجا API Key خودت را وارد کن
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

def call_llm_gpt(prompt: str, model_name: str = "gpt-5.1-2025-11-13"):
    response = client.chat.completions.create(
        model=model_name,   # اینجا اسم مدل واقعی خودت را بگذار
        messages=[
            {"role": "system", "content": "You are a precise NLU classifier."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content


In [29]:
import json
import re

def extract_json_array(text: str):
    """
    متن پاسخ مدل را می‌گیرد و سعی می‌کند اولین JSON array را از آن استخراج و parse کند.
    """
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if not m:
        return None
    candidate = m.group(0)
    try:
        return json.loads(candidate)
    except Exception:
        return None


In [30]:
from tqdm import tqdm

gold_intents = []
pred_intents = []

utterances = llm_df["text"].tolist()
gold_labels = llm_df["intent"].tolist()

BATCH_SIZE = 16  # در هر درخواست 20 جمله می‌فرستیم

for i in tqdm(range(0, len(utterances), BATCH_SIZE)):
    batch_utts = utterances[i:i+BATCH_SIZE]
    batch_gold = gold_labels[i:i+BATCH_SIZE]

    prompt = build_batch_prompt(batch_utts)
    raw_output = call_llm_gpt(prompt, model_name="gpt-5.1-2025-11-13")  # یا مدل GPT-5.1 تو

    js = extract_json_array(raw_output)
    if not js:
        # اگر JSON درست نگرفتیم، کل این batch را None می‌گیریم
        for g in batch_gold:
            gold_intents.append(g)
            pred_intents.append(None)
        continue

    # map از id → پیش‌بینی intent
    pred_map = {}
    for item in js:
        try:
            idx = int(item.get("id", -1))
            intent_raw = normalize_label(item.get("intent", ""))
            if intent_raw in canonical_intents:
                pred_map[idx] = canonical_intents[intent_raw]
            else:
                pred_map[idx] = None
        except Exception:
            continue

    # اضافه‌کردن به لیست نهایی به ترتیب
    for j, g in enumerate(batch_gold, start=1):
        gold_intents.append(g)
        pred_intents.append(pred_map.get(j, None))


100%|██████████| 139/139 [06:29<00:00,  2.80s/it]


In [31]:
valid_count = 0
correct_count = 0

for g, p in zip(gold_intents, pred_intents):
    if p is None:
        continue
    valid_count += 1
    if g == p:
        correct_count += 1

if valid_count > 0:
    intent_accuracy = correct_count / valid_count
else:
    intent_accuracy = 0.0

print("Evaluated on:", valid_count, "examples (LLM returned valid predictions)")
print("Intent Accuracy:", intent_accuracy, "→", intent_accuracy * 100, "%")


Evaluated on: 2211 examples (LLM returned valid predictions)
Intent Accuracy: 0.8141112618724559 → 81.41112618724559 %
